# 使用性檢核:層間相對側向位移角(規範 2.16.1 節)

`seismic_design_2story_8col.ipynb` 裡早就正確定義了這個限制
($0.005$),並留下一句關鍵提示:「此值無法由反曲點法可靠算出,
建議用 `OpenSeesPy` 等有限元素工具驗證」——現在 `Stage 2/3` 正好
有四工具驗證過的模型,補上這一步。

**關鍵細節,先查證過再動手,不是憑印象假設**:規範條文明確指出
檢核層間位移角該用的地震力是 $V_u/2.4$(較小的地震水準,約 `30`
年回歸期),不是直接用 `Stage 2` 用來做強度設計的 `F1_frame`/
`F2_frame`(這組值經追查,已經用 `Fu`/`Ra` 折減過,是設計強度水準,
不是檢核位移角該用的水準)。

In [1]:

try:
    import openseespy.opensees as ops
except ImportError:
    import sys
    !{sys.executable} -m pip install -q "numpy<2.5" openseespy
    import openseespy.opensees as ops

L = 6.0; h1 = 3.5; h2 = 3.5
h_col = 0.4; b_beam, h_beam = 0.3, 0.5
E_rc = 2.463e7
Ic = 0.7*(h_col**4/12); Ib = 0.35*(b_beam*h_beam**3/12)
Ac, Ab = h_col**2, b_beam*h_beam

# Stage 2用的F1_frame/F2_frame是設計強度地震力(已用Fu/Ra折減),
# 規範要求檢核層間位移角要用Vu/2.4這個較小水準
F1_frame, F2_frame = 9.938, 15.900
F1_drift = F1_frame/2.4
F2_drift = F2_frame/2.4
print(f"使用性檢核用的側向力(設計強度/2.4): F1={F1_drift:.3f}kN, F2={F2_drift:.3f}kN")


使用性檢核用的側向力(設計強度/2.4): F1=4.141kN, F2=6.625kN


In [2]:

ops.wipe(); ops.model('basic', '-ndm', 2, '-ndf', 3)
ops.node(1, 0.0, 0.0);   ops.node(2, L, 0.0)
ops.node(3, 0.0, h1);    ops.node(4, L, h1)
ops.node(5, 0.0, h1+h2); ops.node(6, L, h1+h2)
ops.fix(1,1,1,1); ops.fix(2,1,1,1)
ops.geomTransf('Linear', 1)
ops.element('elasticBeamColumn', 1, 1, 3, Ac, E_rc, Ic, 1)
ops.element('elasticBeamColumn', 2, 2, 4, Ac, E_rc, Ic, 1)
ops.element('elasticBeamColumn', 3, 3, 5, Ac, E_rc, Ic, 1)
ops.element('elasticBeamColumn', 4, 4, 6, Ac, E_rc, Ic, 1)
ops.element('elasticBeamColumn', 5, 3, 4, Ab, E_rc, Ib, 1)
ops.element('elasticBeamColumn', 6, 5, 6, Ab, E_rc, Ib, 1)

ops.timeSeries('Linear', 1); ops.pattern('Plain', 1, 1)
ops.load(3, F1_drift, 0.0, 0.0)
ops.load(5, F2_drift, 0.0, 0.0)
ops.system('BandGeneral'); ops.numberer('Plain'); ops.constraints('Plain')
ops.algorithm('Linear'); ops.integrator('LoadControl', 1.0); ops.analysis('Static')
ops.analyze(1)

u3 = ops.nodeDisp(3, 1)
u5 = ops.nodeDisp(5, 1)
print(f"1F樓層水平位移 = {u3*1000:.3f}mm")
print(f"屋頂樓層水平位移 = {u5*1000:.3f}mm")

drift_1F = u3/h1
drift_2F = (u5-u3)/h2
drift_limit = 0.005

print(f"\n1F層間位移角 = {drift_1F:.5f}(規範限制{drift_limit}): "
      f"{'PASS' if drift_1F<=drift_limit else 'FAIL'}(利用率{drift_1F/drift_limit:.1%})")
print(f"2F層間位移角 = {drift_2F:.5f}(規範限制{drift_limit}): "
      f"{'PASS' if drift_2F<=drift_limit else 'FAIL'}(利用率{drift_2F/drift_limit:.1%})")

assert drift_1F <= drift_limit and drift_2F <= drift_limit
print(f"\n[PASS] 使用性檢核(層間位移角)大幅通過, 不是臨界項目")


1F樓層水平位移 = 1.182mm
屋頂樓層水平位移 = 2.621mm

1F層間位移角 = 0.00034(規範限制0.005): PASS(利用率6.8%)
2F層間位移角 = 0.00041(規範限制0.005): PASS(利用率8.2%)

[PASS] 使用性檢核(層間位移角)大幅通過, 不是臨界項目


## 小結

層間位移角大幅通過(利用率僅 `6.8%`/`8.2%`),這根構架的勁度相對這
組側向力來說相當充裕。這一步接續了 `seismic_design_2story_8col.
ipynb` 早就定義好、但一直沒有工具驗證的檢核項目,現在用 `Stage
2` 四工具驗證過的模型完成。

**下一步**(優先順序 `2`):`Stage 7~9` 非線性性能評估,用這次
`design_freeze.json` 的真實配筋重建 `Fiber`/`Hinge`/`Hybrid`
模型。